In [1]:
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone

BASE_URL = "https://api.exchange.coinbase.com"
PRODUCT_ID = "BTC-USD"
GRANULARITY = 86400  
MAX_DAYS = 300       

def iso(dt):
    return dt.astimezone(timezone.utc).isoformat().replace("+00:00", "Z")

def fetch_candles(start, end):
    url = f"{BASE_URL}/products/{PRODUCT_ID}/candles"
    params = {
        "start": iso(start),
        "end": iso(end),
        "granularity": GRANULARITY
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()


end_date = datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0)
start_date = end_date - timedelta(days=10 * 365)
all_candles = []
t = start_date

while t < end_date:
    t_next = min(t + timedelta(days=MAX_DAYS), end_date)
    candles = fetch_candles(t, t_next)
    all_candles.extend(candles)
    t = t_next


df = pd.DataFrame(
    all_candles,
    columns=["timestamp", "low", "high", "open", "close", "volume"]
)

df["date"] = pd.to_datetime(df["timestamp"], unit="s", utc=True).dt.date
df = df.drop_duplicates(subset="date")
df = df.sort_values("date")
df = df[["date", "open", "high", "low", "close", "volume"]]


            date      open      high       low     close        volume
3616  2026-02-02  76895.99  79339.00  74502.21  78666.85  21193.373733
3615  2026-02-03  78666.84  79123.75  72859.00  75661.49  16467.125310
3614  2026-02-04  75661.49  76876.54  71703.72  72998.00  19764.624428
3613  2026-02-05  72999.00  73173.96  62181.65  62791.00  49371.205791
3612  2026-02-06  62788.28  70194.36  60001.00  70064.85  30633.401384


In [4]:
print(df.tail())
print(df.head())
print(f"Total rows: {len(df)}")
df.to_csv("btc_usd_daily.csv", index=False)

            date      open      high       low     close        volume
3616  2026-02-02  76895.99  79339.00  74502.21  78666.85  21193.373733
3615  2026-02-03  78666.84  79123.75  72859.00  75661.49  16467.125310
3614  2026-02-04  75661.49  76876.54  71703.72  72998.00  19764.624428
3613  2026-02-05  72999.00  73173.96  62181.65  62791.00  49371.205791
3612  2026-02-06  62788.28  70194.36  60001.00  70064.85  30633.401384
           date    open    high     low   close        volume
300  2016-02-09  371.02  374.93  370.13  372.68   8424.594208
299  2016-02-10  372.76  383.34  371.87  378.44   9736.036825
298  2016-02-11  378.49  379.50  360.00  378.23  10413.063716
297  2016-02-12  378.23  382.05  377.46  382.05   6162.217623
296  2016-02-13  382.05  396.00  381.99  391.00   5978.856472
Total rows: 3651
